### **Getting to Know a Dataset**

In [ ]:
# 1st step: Initial Exploration: 
           .head() .info() .describe() .shape .value_counts()/("")
------------------------------------------------------------------------------------------
# 2nd step: Data Validation:
           .info() .dtypes 
           DF["var"] = DF["var"].astype(...)            # Updating data types
           DF["var"].isin(["x1", "x2"])                 # Validating categorical data
           ~DF["var"].isin(["x1", "x2"]) 
           DF.select_dtypes("number").head()            # Validating numerical data
------------------------------------------------------------------------------------------
# 3rd step: Data Summarization:
           .groupby() .agg() .pivot_table()
           
           DF.groupby("var1").agg(                      # Example of Named summary columns
              mean_var2=("var2", "mean"), 
              std_var2=("var2", "std"), 
              median_var3=("var3", "median"))

### **Data Cleaning and Imputation**

In [ ]:
# 4th step: Handling the Missing Values"
           DF.isna().sum()                             # Checking for missing values
           
# Why is missing data a problem?
#    1. Affects distributions: Missing heights of taller students
#    2. Less representative of the populationCertain groups disproportionately represented
#    3. Can result in drawing incorrect conclusions

![My Chart](../images/5.png)

In [ ]:
# Droping missing values
threshold = len(DF) * 0.05 
cols_to_drop = DF.columns[DF.isna().sum() <= threshold] 
DF.dropna(subset=cols_to_drop, inplace=True) 

# Imputing mean, median, mode
cols_with_missing_values = DF.columns[DF.isna().sum() > 0] 
for col in cols_with_missing_values[:-1]: 
    DF[col].fillna(DF[col].mode()[0])

# Imputing by sub-group
DF_dict = DF.groupby("var1")["var2"].median().to_dict()       # or mode or mean or .agg()
DF["var2"] = DF["var2"].fillna(DF["var1"].map(DF_dict)) 
------------------------------------------------------------------------------------------
# 5th step: Working with Categorical Data (Converting and Analyzing):
           DF["var"].value_counts()
           DF["var"].value_counts(normalize=True)       # Relative class frequency 
           DF["var"].nunique()
           DF["var"].str.contains("something")          # Extracting value from categories
                                 ("smth1|smth2") ("^something")

           # Creating the categorical column
           conditions = ((DF["var"].str.contains("smth1")),                  
                         (DF["var"].str.contains("smth2")),  
                         (DF["var"].str.contains("smth3")), ...)
           DF["NEW_VAR_CATEGORIES"] = np.select(conditions, var, default="Other")
------------------------------------------------------------------------------------------
# 6th step: Working with Numeric Data:
           # Converting strings to numbers
           DF["var"].str.replace("characters to remove","characters to replace them with")
           DF["var"].astype(float)

           # Adding summary statistics into a DataFrame (example: std)
           DF["std_dev"] = salaries.groupby("var1")["var2"].transform(lambda x: x.std())
------------------------------------------------------------------------------------------
# 7th step: Handling Outliers
           iqr = DF["var"].quantile(0.75) - DF["var"].quantile(0.25)
           upper = DF["var"].quantile(0.75) + (1.5 * iqr)
           lower = DF["var"].quantile(0.25) - (1.5 * iqr)

           # Subsetting the Outliers
           DF[(DF["var"] < lower) | (DF["var"] > upper)] \ [["var1", "var2", "var3", ...]]

           # Dropping the Outliers(if necessary)
           DF[(DF["var"] > lower) & (DF["var"] < upper)] 
------------------------------------------------------------------------------------------

### **Relationships in Data**

In [ ]:
# 8th step: Patterns over time
           DF = pd.read_csv("DF.csv", parse_dates=["DateTimeVAR"]) 
           DF["DateTimeVAR"] = pd.to_datetime(DF["DateTimeVAR"])

           # Extract parts of a full date: dt.month, dt.day, and dt.year dt.weekday etc.
           DF["DateTimeVAR"] = divorce["DateTimeVAR"].dt.month 
------------------------------------------------------------------------------------------
# 9th step: Correlation
           DF["var1"].corr(DF["var2"])                          # between 2 variables
           DF.corr(numeric_only=True)                           # Matrix of Correlations
           sns.heatmap(DF.corr(numeric_only=True), annot=True)  # Heatmap plot
           sns.pairplot(data=DF, vars=["var1", "var2", ...])    # Pairplot (INTRESTING)
           
           # Kernel Density Estimate (KDE) plots (normal and cumulative)
           sns.kdeplot(data=divorce, x="var", hue="var1", cut=0)# To start from zero only
           sns.kdeplot(data=divorce, x="var", hue="var1", cut=0, cumulative=True) 

### **Turning Exploratory Analysis into Action**

In [ ]:
pd.crosstab(DF["var1_isindex"], DF["var2_iscolumn"])     # cross-tabular matrix
pd.crosstab(DF["var1_isindex"], DF["var2_iscolumn"],     # Aggregated values with crosstab
            values=planes["varrrr"], aggfunc="median")

# NOTE: GENERATING NEW FEATURES: If in the first glance you can not see enough correlations in your dataset, maybe changing the datatypes to float could solve the problems.(see the step 8)
twenty_fifth = DF["var"].quantile(0.25)                                          # EXAMPLE
median = DF["var"].median() 
seventy_fifth = DF["var"].quantile(0.75) 
maximum = DF["var"].max() 

labels = ["Economy", "Premium Economy", "Business Class", "First Class"] 
bins = [0, twenty_fifth, median, seventy_fifth, maximum]

DF["NEW-VAR"] = pd.cut(DF["var"], labels=labels, bins=bins)
-------------------------------------------------------------------------------------------
# Another exapmle of super important "pd.cut()" for categorization:
crimes["victim_ages"] = pd.cut(crimes["Vict Age"],
                               bins=[-1, 17, 25, 34, 44, 54, 64, float("inf")],
                               labels=victim_ages)
# NOTE: Why -1? Because we want ages 0 through 17 to be in the first group. Since pd.cut() uses boundaries, -1 gives us a boundary below 0.
#       Why float("inf")? inf means infinity.